In [1]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction

# Structure Refinement: Co2SiO4, D20 (T-scan)

This example demonstrates a Rietveld refinement of the Co2SiO4 crystal
structure using constant-wavelength neutron powder diffraction data
from D20 at ILL. A sequential refinement is performed against a
temperature scan using sequential fitting, which processes each data
file independently without loading all datasets into memory at once.

## 🛠️ Import Library

In [2]:
import easydiffraction as edi

## 📦 Define Project

The project object manages structures, experiments, analysis, display,
and other related components.

In [3]:
project = edi.Project(name='cosio_d20_scan')
analysis = project.analysis
display = project.display

The project must be saved before running sequential fitting, so that
results can be written to `analysis/results.csv`.

In [4]:
project.save_as(dir_path='projects/refine-cosio-d20-tscan')

Saving project 📦 'cosio_d20_scan' to '../../../projects/refine-cosio-d20-tscan'


├── 📄 project.edi
├── 📁 structures/
├── 📁 experiments/
├── 📁 analysis/
│   └── 📄 analysis.edi
└── 📁 reports/
    └── 📄 cosio_d20_scan.html


## 🧩 Define Structure

This section shows how to add structures and modify their
parameters.

### Create Structure

In [5]:
project.structures.create(name='cosio')
struct = project.structures['cosio']

### Set Space Group

In [6]:
struct.space_group.name_h_m = 'P n m a'
struct.space_group.coord_system_code = 'abc'

### Set Unit Cell

In [7]:
struct.cell.length_a = 10.31
struct.cell.length_b = 6.0
struct.cell.length_c = 4.79

### Set Atom Sites

In [8]:
struct.atom_sites.create(
    id='Co1',
    type_symbol='Co',
    fract_x=0,
    fract_y=0,
    fract_z=0,
    adp_iso=0.3,
)
struct.atom_sites.create(
    id='Co2',
    type_symbol='Co',
    fract_x=0.279,
    fract_y=0.25,
    fract_z=0.985,
    adp_iso=0.3,
)
struct.atom_sites.create(
    id='Si',
    type_symbol='Si',
    fract_x=0.094,
    fract_y=0.25,
    fract_z=0.429,
    adp_iso=0.34,
)
struct.atom_sites.create(
    id='O1',
    type_symbol='O',
    fract_x=0.091,
    fract_y=0.25,
    fract_z=0.771,
    adp_iso=0.63,
)
struct.atom_sites.create(
    id='O2',
    type_symbol='O',
    fract_x=0.448,
    fract_y=0.25,
    fract_z=0.217,
    adp_iso=0.59,
)
struct.atom_sites.create(
    id='O3',
    type_symbol='O',
    fract_x=0.164,
    fract_y=0.032,
    fract_z=0.28,
    adp_iso=0.83,
)

### Display Structure

In [9]:
project.structure_style.atom_view = 'adp'
project.display.structure(struct_name='cosio')

Structure 🧩 'cosio' (Atom view type: 'adp')


## 🔬 Define Experiment

For sequential fitting, we create a single template experiment from
the first data file. This template defines the instrument, peak
profile, background, and linked structures that will be reused for every
data file in the scan.

### Download Data

In [10]:
zip_path = edi.download_data('meas-cosio-d20-scan-3f', destination='data')

Getting data...


Data 'meas-cosio-d20-scan-3f': Co2SiO4, D20 (ILL), 3 files: ~50K, ~300K, ~500K


✅ Data 'meas-cosio-d20-scan-3f' downloaded to '../../../data/meas-cosio-d20-scan-3f.zip'


### Extract Data Files

In [11]:
scan_data_dir = 'experiments/d20_scan'
data_paths = edi.extract_data_paths_from_zip(
    zip_path,
    destination=project.metadata.path / scan_data_dir,
)

### Create Template Experiment

In [12]:
project.experiments.add_from_data_path(
    name='d20',
    data_path=data_paths[0],
)
expt = project.experiments['d20']

Data loaded successfully


Experiment 🔬 'd20'. Number of data points: 1507.


### Set Instrument

In [13]:
expt.instrument.setup_wavelength = 1.87
expt.instrument.calib_twotheta_offset = 0.29

### Set Peak Profile

In [14]:
expt.peak.broad_gauss_u = 0.24
expt.peak.broad_gauss_v = -0.53
expt.peak.broad_gauss_w = 0.38
expt.peak.broad_lorentz_y = 0.02
expt.peak.cutoff_fwhm = 8

### Set Excluded Regions

In [15]:
expt.excluded_regions.create(id='1', start=0, end=8)
expt.excluded_regions.create(id='2', start=150, end=180)

### Set Background

In [16]:
expt.background.create(id='1', position=8, intensity=609)
expt.background.create(id='2', position=9, intensity=581)
expt.background.create(id='3', position=10, intensity=563)
expt.background.create(id='4', position=11, intensity=540)
expt.background.create(id='5', position=12, intensity=520)
expt.background.create(id='6', position=15, intensity=507)
expt.background.create(id='7', position=25, intensity=463)
expt.background.create(id='8', position=30, intensity=434)
expt.background.create(id='9', position=50, intensity=451)
expt.background.create(id='10', position=70, intensity=431)
expt.background.create(id='11', position=90, intensity=414)
expt.background.create(id='12', position=110, intensity=361)
expt.background.create(id='13', position=130, intensity=292)
expt.background.create(id='14', position=150, intensity=241)

### Set Linked Structures

In [17]:
expt.linked_structures.create(structure_id='cosio', scale=1.2)

## 🚀 Perform Analysis

This section shows how to set free parameters, define constraints,
and run the sequential refinement.

### Set Free Parameters

In [18]:
struct.cell.length_a.free = True
struct.cell.length_b.free = True
struct.cell.length_c.free = True

struct.atom_sites['Co2'].fract_x.free = True
struct.atom_sites['Co2'].fract_z.free = True
struct.atom_sites['Si'].fract_x.free = True
struct.atom_sites['Si'].fract_z.free = True
struct.atom_sites['O1'].fract_x.free = True
struct.atom_sites['O1'].fract_z.free = True
struct.atom_sites['O2'].fract_x.free = True
struct.atom_sites['O2'].fract_z.free = True
struct.atom_sites['O3'].fract_x.free = True
struct.atom_sites['O3'].fract_y.free = True
struct.atom_sites['O3'].fract_z.free = True

struct.atom_sites['Co1'].adp_iso.free = True
struct.atom_sites['Co2'].adp_iso.free = True
struct.atom_sites['Si'].adp_iso.free = True
struct.atom_sites['O1'].adp_iso.free = True
struct.atom_sites['O2'].adp_iso.free = True
struct.atom_sites['O3'].adp_iso.free = True

In [19]:
expt.linked_structures['cosio'].scale.free = True

expt.instrument.calib_twotheta_offset.free = True

expt.peak.broad_gauss_u.free = True
expt.peak.broad_gauss_v.free = True
expt.peak.broad_gauss_w.free = True
expt.peak.broad_lorentz_y.free = True

for point in expt.background:
    point.intensity.free = True

### Set Constraints

Set aliases for parameters.

In [20]:
analysis.aliases.create(
    id='biso_Co1',
    param=struct.atom_sites['Co1'].adp_iso,
)
analysis.aliases.create(
    id='biso_Co2',
    param=struct.atom_sites['Co2'].adp_iso,
)

Set constraints.

In [21]:
analysis.constraints.create(expression='biso_Co2 = biso_Co1')

### Set Minimizer

In [22]:
analysis.minimizer.type = 'bumps (lm)'

⚠️ Switching minimizer type removes these settings:
• gradient_tolerance


Current minimizer changed to


bumps (lm)


### Run Fitting

This is the fitting of the first dataset to optimize the initial
parameters for the sequential fitting. This step is optional but can
help with convergence and speed of the sequential fitting, especially
if the initial parameters are far from optimal.

In [23]:
analysis.minimizer.chi_square_change_tolerance = 1e-2

In [24]:
analysis.fit()

<IPython.core.display.Javascript object>

Standard fitting


📋 Using experiment 🔬 'd20' for 'single' fitting


🚀 Starting fit process with 'bumps (lm)'...


📈 Goodness-of-fit progress:


,iteration,time (s),χ²,change / status
1,1,0.05,12.77,
2,41,2.91,5.11,60.0% ↓
3,81,5.95,4.82,5.6% ↓
4,123,20.06,4.82,


🏆 Best goodness-of-fit (reduced χ²) is 4.82 at iteration 123


✅ Fitting complete.


In [25]:
display.fit.results()

⚙️ Settings used:


,Name,Value,Description
1,max_iterations,1000,Maximum solver iterations.
2,chi_square_change_tolerance,0.01,Relative change in the objective (chi-square) used to stop fitting.
3,parameter_change_tolerance,1e-08,Relative change in fitted parameters used to stop fitting.


📋 Least-squares fit results:


,Metric,Value
1,🧪 Minimizer,bumps (lm)
2,✅ Overall status,success
3,⏱️ Fitting time (seconds),20.06
4,📏 Goodness-of-fit (reduced χ²),4.82
5,"📏 R-factor (Rf, %)",3.16
6,"📏 R-factor squared (Rf², %)",4.68
7,"📏 Weighted R-factor (wR, %)",4.14


📈 Refined parameters:


,datablock,category,entry,parameter,units,start,value,s.u.,change
1,cosio,cell,,length_a,Å,10.3100,10.3071,0.0003,0.03 % ↓
2,cosio,cell,,length_b,Å,6.0000,6.0030,0.0002,0.05 % ↑
3,cosio,cell,,length_c,Å,4.7900,4.7865,0.0001,0.07 % ↓
4,cosio,atom_site,Co1,adp_iso,Å²,0.3000,0.1578,0.0806,47.41 % ↓
5,cosio,atom_site,Co2,fract_x,,0.2790,0.2784,0.0007,0.22 % ↓
6,cosio,atom_site,Co2,fract_z,,0.9850,0.9810,0.0015,0.41 % ↓
7,cosio,atom_site,Si,fract_x,,0.0940,0.0934,0.0004,0.62 % ↓
8,cosio,atom_site,Si,fract_z,,0.4290,0.4285,0.0009,0.12 % ↓
9,cosio,atom_site,Si,adp_iso,Å²,0.3400,0.3834,0.0649,12.77 % ↑
10,cosio,atom_site,O1,fract_x,,0.0910,0.0907,0.0003,0.38 % ↓


### Display Correlations

In [26]:
display.fit.correlations()

### Display Pattern

In [27]:
display.pattern(expt_name='d20')

### Display Structure

In [28]:
project.structure_style.atom_view = 'adp'
project.display.structure(struct_name='cosio')

Structure 🧩 'cosio' (Atom view type: 'adp')


### Run Sequential Fitting

Set output verbosity level to "short" to show only one-line status
messages during the analysis process.

In [29]:
project.verbosity = 'short'


Create a persisted extract rule that reads the temperature from each
data file.

In [30]:
temperature = 'diffrn.ambient_temperature'

In [31]:
analysis.sequential_fit_extract.create(
    id='temperature',
    target=temperature,
    pattern=r'^TEMP\s+([0-9.]+)',
    required=True,
)

Set the sequential fitting parameters.

In [32]:
analysis.fitting_mode.type = 'sequential'
analysis.sequential_fit.data_dir = scan_data_dir
analysis.sequential_fit.max_workers = 'auto'
analysis.sequential_fit.reverse = True

Fitting mode changed to


sequential


Run the sequential fit over all data files in the scan directory.

In [33]:
analysis.fit()

<IPython.core.display.Javascript object>

Sequential fitting


🚀 Starting fit process with 'bumps (lm)'...


📋 3 files in 1 chunks (max_workers=4)


📈 Goodness-of-fit progress:


,chunk,progress,time (s),files,count,average χ²,status
1,1/1,100.0%,118.44,all594842.dat - all594687.dat,3,4.37,✅


✅ Sequential fitting complete: 3 files processed.


📄 Results saved to '../../../projects/refine-cosio-d20-tscan/analysis/results.csv'


### Replay a Dataset

Apply fitted parameters from the first CSV row and plot the result.

In [34]:
project.apply_params_from_csv(row_index=0)
display.pattern(expt_name='d20')


Apply fitted parameters from the last CSV row and plot the result.

In [35]:
project.apply_params_from_csv(row_index=-1)
display.pattern(expt_name='d20')

### Display Parameter Evolution

Reuse the extracted diffrn path as the x-axis in the following plots.

Plot fit quality metrics vs. temperature.

In [36]:
display.fit.series(analysis.fit_result.success, versus=temperature)
display.fit.series(analysis.fit_result.reduced_chi_square, versus=temperature)
display.fit.series(analysis.fit_result.iterations, versus=temperature)

Plot unit cell parameters vs. temperature.

In [37]:
display.fit.series(struct.cell.length_a, versus=temperature)
display.fit.series(struct.cell.length_b, versus=temperature)
display.fit.series(struct.cell.length_c, versus=temperature)

Plot isotropic displacement parameters vs. temperature.

In [38]:
display.fit.series(struct.atom_sites['Co1'].adp_iso, versus=temperature)
display.fit.series(struct.atom_sites['Si'].adp_iso, versus=temperature)
display.fit.series(struct.atom_sites['O1'].adp_iso, versus=temperature)
display.fit.series(struct.atom_sites['O2'].adp_iso, versus=temperature)
display.fit.series(struct.atom_sites['O3'].adp_iso, versus=temperature)

Plot selected fractional coordinates vs. temperature.

In [39]:
display.fit.series(struct.atom_sites['Co2'].fract_x, versus=temperature)
display.fit.series(struct.atom_sites['Co2'].fract_z, versus=temperature)
display.fit.series(struct.atom_sites['O1'].fract_z, versus=temperature)
display.fit.series(struct.atom_sites['O2'].fract_z, versus=temperature)
display.fit.series(struct.atom_sites['O3'].fract_z, versus=temperature)

## 💾 Save Project

Save the fitted parameters and analysis results.

In [40]:
project.save()

Saving project 📦 'cosio_d20_scan' to '../../../projects/refine-cosio-d20-tscan'


├── 📄 project.edi
├── 📁 structures/
│   └── 📄 cosio.edi
├── 📁 experiments/
│   └── 📄 d20.edi
├── 📁 analysis/
│   ├── 📄 analysis.edi
│   └── 📄 results.csv
└── 📁 reports/
    └── 📄 cosio_d20_scan.html
